<a href="https://colab.research.google.com/github/Yashika-Bayeen/agentic-ai/blob/main/Day_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libraries
!pip install langchain langchain-groq python-dotenv -q
print("✅ Libraries installed.")

In [ ]:
import re
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Tool definition
def calculate(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": None}, {}))
    except Exception as e:
        return f"Error evaluating: {e}"

REACT_PROMPT = """You are an agent. Solve the problem by stepping through thoughts and actions.
You have access to a tool: calculate(expression).

Format your output EXACTLY as follows:
Thought: Describe your thinking process.
Action: calculate(your_math_expression)
Observation: The tool result will appear here.
... (Repeat Thought and Action if needed)
Answer: The final result of the calculation.

Begin!\nQuestion: """
print("✅ ReAct system ready.")

In [ ]:
def run_react_agent(query, max_turns=5):
    prompt = REACT_PROMPT + query
    history = ""

    for turn in range(max_turns):
        print(f"\n--- Turn {turn+1} ---")
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt + history}],
            temperature=0.0
        )
        content = response.choices[0].message.content
        print(content)

        if "Answer:" in content:
            break

        action_match = re.search(r"Action:\s*calculate\((.*?)\)", content)
        if action_match:
            expression = action_match.group(1)
            observation = calculate(expression)
            print(f"\nObservation: {observation}")
            history += f"\n{content}\nObservation: {observation}"
        else:
            print("\nCould not parse Action.")
            break

run_react_agent("What is 4871 multiplied by 209?")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"))
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly counselor. Speak in short replies."),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

reply = chain.invoke({"question": "What is LangChain in one sentence?"})
print("LangChain reply:", reply)

In [ ]:
def division_calc(expression: str) -> str:
    if "/" in expression:
        num, denom = expression.split("/")
        if float(denom) == 0.0:
            raise ZeroDivisionError("Calculation failed: Cannot divide by zero!")
    return str(eval(expression))

def run_self_correcting_agent(query, max_turns=5):
    prompt = """You are an agent that uses division_calc(expression).
If you receive an error observation, you MUST reason about the error, adjust your input and try again.
Format:
Thought: ...
Action: division_calc(expression)
Observation: ...
Answer: ...\nQuestion: """ + query

    history = ""

    for turn in range(max_turns):
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt + history}],
            temperature=0.0
        )
        content = response.choices[0].message.content
        print(f"\n[Agent Output]:\n{content}")

        if "Answer:" in content:
            break

        action_match = re.search(r"Action:\s*division_calc\((.*?)\)", content)
        if action_match:
            expr = action_match.group(1)
            try:
                observation = division_calc(expr)
            except Exception as e:
                observation = f"ERROR: {e}"

            print(f"\n[Observation]: {observation}")
            history += f"\n{content}\nObservation: {observation}"

run_self_correcting_agent("Divide 10 by (5 minus 5). If it fails, add 1 to denominator and calculate.")

In [ ]:
import datetime

def save_agent_trace(query, history_str, filepath="agent_trace.md"):
    md_content = f"""# Agent Execution Trace Report
**Timestamp:** {datetime.datetime.now().isoformat()}
**User Query:** {query}

## Reasoning Pathway
"""
    steps = history_str.split("Observation:")
    for i, step in enumerate(steps):
        md_content += f"### Step {i+1}\n"
        md_content += f"```\n{step.strip()}\n```\n\n"

    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(md_content)
    print(f"Saved audit report trace to {filepath}")

mock_history = "Thought: Need to compute 4871*209\nAction: calculate(4871*209)\nObservation: 1018039\nThought: Done"
save_agent_trace("Compute 4871*209", mock_history)

In [ ]:
from langchain_core.exceptions import OutputParserException
import json

class AutoCorrectionParser(StrOutputParser):
    def parse(self, text: str) -> dict:
        cleaned = text.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        cleaned = cleaned.strip()

        try:
            return json.loads(cleaned)
        except json.JSONDecodeError:
            print("Failed to parse JSON. Applying auto-recovery output wrapper.")
            return {"status": "error", "raw_output": text}

# Test custom parser
parser = AutoCorrectionParser()
print("Standard parse:", parser.parse('{"value": 10}'))
print("Recovered parse:", parser.parse('Calculated sum is 50. Output could not parse.'))